In [ ]:
import pandas as pd
import numpy as np

from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings("ignore")


# ============================================================
# LOAD THE CORRECTED MASTER FEATURE TABLE
# ============================================================

clean_fm = pd.read_pickle("clean_fm_clusters.pkl")

CLUSTER_ORDER = [
    "Peak-Exposed",
    "Typical Household",
    "Heavy Continuous",
    "Daytime-Active"
]

CORE_FEATURES = [
    "evening_peak_ratio",
    "night_baseload_ratio",
    "load_uniformity",
    "load_factor",
    "leisure_usage_ratio",
    "log_daily_kwh"
]


# ============================================================
# DYNAMIC ARCHETYPE MAPPING
# ============================================================

def infer_archetype_name_map(feature_table, label_col="final_cluster"):
    """
    Infer stable archetype names from the locked feature profiles.

    This avoids dependence on arbitrary KMeans numeric cluster IDs.
    """
    required_columns = [
        label_col,
        "evening_peak_ratio",
        "night_baseload_ratio",
        "load_factor",
        "log_daily_kwh"
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in feature_table.columns
    ]

    if missing_columns:
        raise RuntimeError(
            "Cannot infer archetype names because required columns are missing: "
            f"{missing_columns}"
        )

    profile_medians = (
        feature_table
        .groupby(label_col)[
            [
                "evening_peak_ratio",
                "night_baseload_ratio",
                "load_factor",
                "log_daily_kwh"
            ]
        ]
        .median()
    )

    if len(profile_medians) != 4:
        raise RuntimeError(
            "The stored master partition must contain exactly four clusters."
        )

    remaining_ids = set(profile_medians.index.tolist())
    mapping = {}

    heavy_id = profile_medians["load_factor"].idxmax()
    mapping[heavy_id] = "Heavy Continuous"
    remaining_ids.remove(heavy_id)

    peak_id = profile_medians.loc[
        list(remaining_ids),
        "evening_peak_ratio"
    ].idxmax()
    mapping[peak_id] = "Peak-Exposed"
    remaining_ids.remove(peak_id)

    daytime_id = profile_medians.loc[
        list(remaining_ids),
        "night_baseload_ratio"
    ].idxmin()
    mapping[daytime_id] = "Daytime-Active"
    remaining_ids.remove(daytime_id)

    typical_id = list(remaining_ids)[0]
    mapping[typical_id] = "Typical Household"

    return dict(sorted(mapping.items(), key=lambda item: item[0]))


CLUSTER_NAMES = infer_archetype_name_map(
    clean_fm,
    label_col="final_cluster"
)

CLUSTER_NAME_TO_ID = {
    cluster_name: cluster_id
    for cluster_id, cluster_name in CLUSTER_NAMES.items()
}

CLUSTER_IDS = [
    CLUSTER_NAME_TO_ID[name]
    for name in CLUSTER_ORDER
]

print("Dynamic cluster-name map inferred from locked feature medians:")
for cluster_id, cluster_name in CLUSTER_NAMES.items():
    print(f"  {cluster_id} -> {cluster_name}")


# ============================================================
# PREPARE THE LOCKED FEATURE REPRESENTATION
# ============================================================

missing_features = [
    feature
    for feature in CORE_FEATURES
    if feature not in clean_fm.columns
]

if missing_features:
    raise RuntimeError(
        f"Missing required clustering features: {missing_features}"
    )

X = clean_fm[CORE_FEATURES].to_numpy(dtype=float)
original_labels = clean_fm["final_cluster"].to_numpy(dtype=int)

if not np.isfinite(X).all():
    raise RuntimeError(
        "The clustering feature matrix contains missing or infinite values."
    )

if pd.Series(original_labels).nunique() != 4:
    raise RuntimeError(
        "The stored master labels must contain exactly four clusters."
    )

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("\nFeature ranges after standardization:")
display(
    pd.DataFrame(
        X_scaled,
        columns=CORE_FEATURES
    )
    .describe()
    .round(3)
)


# ============================================================
# STEP 1: SEED STABILITY AND LOCKED-SEED REPRODUCTION
# ============================================================

print("\n" + "=" * 70)
print("  STEP 1: SEED STABILITY AND LOCKED-SEED REPRODUCTION")
print("=" * 70)

tested_seeds = [
    0, 1, 2, 3, 4, 5,
    10, 21, 42, 99, 100, 123, 999
]

seed_ari_results = {}
seed_cluster_sizes = {}

for seed in tested_seeds:
    model = KMeans(
        n_clusters=4,
        random_state=seed,
        n_init=20,
        max_iter=500
    )

    labels = model.fit_predict(X_scaled)

    ari = adjusted_rand_score(
        original_labels,
        labels
    )

    sizes = sorted(
        pd.Series(labels)
        .value_counts()
        .tolist(),
        reverse=True
    )

    seed_ari_results[seed] = float(ari)
    seed_cluster_sizes[seed] = sizes

    print(
        f"  Seed {seed:>4}: "
        f"ARI={ari:.6f}  "
        f"sizes={sizes}"
    )

locked_seed_ari = seed_ari_results[42]

exact_reproduction_seeds = [
    seed
    for seed, ari in seed_ari_results.items()
    if np.isclose(ari, 1.0, atol=1e-12)
]

mean_seed_ari = float(
    np.mean(list(seed_ari_results.values()))
)

minimum_seed_ari = float(
    np.min(list(seed_ari_results.values()))
)

print(
    f"\nLocked seed 42 ARI: "
    f"{locked_seed_ari:.6f}"
)

print(
    f"Mean tested-seed ARI: "
    f"{mean_seed_ari:.6f}"
)

print(
    f"Minimum tested-seed ARI: "
    f"{minimum_seed_ari:.6f}"
)

print(
    "Seeds producing exact label-equivalent partitions:",
    exact_reproduction_seeds
)

assert np.isclose(
    locked_seed_ari,
    1.0,
    atol=1e-12
), (
    "The locked seed 42 did not reproduce the stored master partition."
)


# ============================================================
# STEP 2: ALGORITHM COMPARISON
# ============================================================

print("\n" + "=" * 70)
print("  STEP 2: ALGORITHM COMPARISON")
print("=" * 70)

print(
    "The stored master KMeans labels are used as the reference partition."
)


# ------------------------------------------------------------
# Gaussian Mixture Model
# ------------------------------------------------------------

gmm = GaussianMixture(
    n_components=4,
    covariance_type="full",
    random_state=42,
    n_init=20,
    max_iter=1000,
    reg_covar=1e-6
)

gmm_labels = gmm.fit_predict(X_scaled)

if not gmm.converged_:
    raise RuntimeError(
        f"GMM did not converge after {gmm.n_iter_} iterations."
    )

print(
    f"\nGMM converged after {gmm.n_iter_} iterations."
)


# ------------------------------------------------------------
# Ward hierarchical clustering
# ------------------------------------------------------------

hierarchical_model = AgglomerativeClustering(
    n_clusters=4,
    linkage="ward"
)

hierarchical_labels = hierarchical_model.fit_predict(X_scaled)


# ============================================================
# COMPARISON METRICS
# ============================================================

ari_km_gmm = adjusted_rand_score(
    original_labels,
    gmm_labels
)

ari_km_hier = adjusted_rand_score(
    original_labels,
    hierarchical_labels
)

ari_gmm_hier = adjusted_rand_score(
    gmm_labels,
    hierarchical_labels
)

sil_km = silhouette_score(
    X_scaled,
    original_labels
)

sil_gmm = silhouette_score(
    X_scaled,
    gmm_labels
)

sil_hier = silhouette_score(
    X_scaled,
    hierarchical_labels
)

print("\nAdjusted Rand Index comparisons:")
print(f"  Stored KMeans vs GMM:          {ari_km_gmm:.3f}")
print(f"  Stored KMeans vs Ward:         {ari_km_hier:.3f}")
print(f"  GMM vs Ward:                   {ari_gmm_hier:.3f}")

print("\nSilhouette scores:")
print(f"  Stored KMeans:                 {sil_km:.3f}")
print(f"  GMM:                           {sil_gmm:.3f}")
print(f"  Ward hierarchical:             {sil_hier:.3f}")

print("\nCluster sizes:")
print(
    "  Stored KMeans: ",
    sorted(
        pd.Series(original_labels)
        .value_counts()
        .tolist(),
        reverse=True
    )
)
print(
    "  GMM:           ",
    sorted(
        pd.Series(gmm_labels)
        .value_counts()
        .tolist(),
        reverse=True
    )
)
print(
    "  Ward:          ",
    sorted(
        pd.Series(hierarchical_labels)
        .value_counts()
        .tolist(),
        reverse=True
    )
)


# ============================================================
# CONSERVATIVE INTERPRETATION
# ============================================================

print("\n" + "=" * 70)
print("  CONSERVATIVE INTERPRETATION")
print("=" * 70)

print(f"""
Seed stability:

The locked KMeans specification using random_state=42, n_init=20 and the
standardized six-feature matrix reproduced the stored master partition
exactly, with ARI={locked_seed_ari:.3f}. Multiple tested seeds also produced
identical or near-identical partitions, supporting strong initialization
stability.

KMeans versus GMM:

The ARI of {ari_km_gmm:.3f} indicates that KMeans and GMM identify different
partition structures in the same six-feature representation. This reflects
sensitivity to the clustering objective. KMeans minimizes within-cluster
squared Euclidean distances, whereas GMM estimates a probabilistic mixture
distribution.

KMeans achieved a higher silhouette score than GMM in this comparison
({sil_km:.3f} versus {sil_gmm:.3f}). This supports KMeans for the study's
compact geometric segmentation objective, but it does not invalidate the
GMM partition.

KMeans versus Ward hierarchical clustering:

The ARI of {ari_km_hier:.3f} indicates that Ward hierarchical clustering also
produced a substantially different partition. Its silhouette score was lower
than that of KMeans ({sil_hier:.3f} versus {sil_km:.3f}), indicating weaker
separation under the same standardized feature representation.

Overall:

KMeans is retained because it provides the strongest combination of
silhouette performance, initialization stability, interpretability and
consistency with the study's behavioural segmentation objective.
Cross-algorithm disagreement is reported as algorithm sensitivity.

No claim of global optimality is made, and the alternative partitions are not
described as invalid.
""")


In [ ]:
silhouette_comparison = pd.DataFrame({
    "algorithm": [
        "Stored KMeans",
        "Gaussian Mixture Model",
        "Ward hierarchical"
    ],
    "silhouette": [
        sil_km,
        sil_gmm,
        sil_hier
    ]
}).sort_values(
    "silhouette",
    ascending=False
).reset_index(drop=True)

print("Silhouette comparison under the same standardized feature space:")
display(
    silhouette_comparison.round(3)
)


In [ ]:
print("=" * 70)
print("  SANITY CHECK SUMMARY")
print("=" * 70)

print(f"""
CONFIRMED FROM THIS INDEPENDENT CHECK:

1. REPRODUCIBILITY:

   The locked K=4 KMeans specification used the six-feature matrix,
   StandardScaler, random_state=42 and n_init=20.

   This specification reproduced the stored master labels exactly:

       Locked seed 42 ARI = {locked_seed_ari:.3f}

   Across the tested seeds:

       Mean ARI = {mean_seed_ari:.3f}
       Minimum ARI = {minimum_seed_ari:.3f}

   Several seeds generated label-equivalent partitions, supporting strong
   initialization stability.

2. ALGORITHM COMPARISON:

   Under the same standardized six-feature representation:

       KMeans silhouette = {sil_km:.3f}
       GMM silhouette = {sil_gmm:.3f}
       Ward silhouette = {sil_hier:.3f}

   KMeans therefore achieved the strongest internal separation among these
   three tested algorithms for the study's geometric segmentation objective.

3. CROSS-ALGORITHM SENSITIVITY:

   KMeans-GMM ARI = {ari_km_gmm:.3f}
   KMeans-Ward ARI = {ari_km_hier:.3f}

   These values demonstrate sensitivity to the clustering objective.
   They do not invalidate the alternative algorithms or indicate a failure
   of the final KMeans solution.

SAFE TO REPORT:

  - the locked seed 42 reproduces the stored master partition exactly;
  - initialization stability is strong across the tested seeds;
  - KMeans achieved the highest silhouette among the tested algorithms;
  - alternative algorithms identify different partition structures.

DO NOT CLAIM:

  - global optimality;
  - universally optimal clusters;
  - that GMM or hierarchical clustering are invalid;
  - that low cross-algorithm ARI proves model failure.
""")


In [ ]:
import pandas as pd
import numpy as np
import glob
import warnings

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from scipy.stats import fisher_exact

warnings.filterwarnings("ignore")


# ============================================================
# LOAD CORRECTED MASTER FEATURE TABLE
# ============================================================

clean_fm = pd.read_pickle(
    "clean_fm_clusters.pkl"
)

CORE_FEATURES = [
    "evening_peak_ratio",
    "night_baseload_ratio",
    "load_uniformity",
    "load_factor",
    "leisure_usage_ratio",
    "log_daily_kwh"
]


# ============================================================
# REBUILD DYNAMIC ARCHETYPE MAPPING WHEN REQUIRED
# ============================================================

if "infer_archetype_name_map" not in globals():

    def infer_archetype_name_map(
        feature_table,
        label_col="final_cluster"
    ):
        """
        Infer stable archetype names from locked feature medians.
        """
        required_columns = [
            label_col,
            "evening_peak_ratio",
            "night_baseload_ratio",
            "load_factor",
            "log_daily_kwh"
        ]

        missing_columns = [
            column
            for column in required_columns
            if column not in feature_table.columns
        ]

        if missing_columns:
            raise RuntimeError(
                "Cannot infer archetype names because required columns are "
                f"missing: {missing_columns}"
            )

        medians = (
            feature_table
            .groupby(label_col)[
                [
                    "evening_peak_ratio",
                    "night_baseload_ratio",
                    "load_factor",
                    "log_daily_kwh"
                ]
            ]
            .median()
        )

        if len(medians) != 4:
            raise RuntimeError(
                "Expected exactly four stored master clusters."
            )

        remaining_ids = set(medians.index.tolist())
        mapping = {}

        heavy_id = medians["load_factor"].idxmax()
        mapping[heavy_id] = "Heavy Continuous"
        remaining_ids.remove(heavy_id)

        peak_id = medians.loc[
            list(remaining_ids),
            "evening_peak_ratio"
        ].idxmax()
        mapping[peak_id] = "Peak-Exposed"
        remaining_ids.remove(peak_id)

        daytime_id = medians.loc[
            list(remaining_ids),
            "night_baseload_ratio"
        ].idxmin()
        mapping[daytime_id] = "Daytime-Active"
        remaining_ids.remove(daytime_id)

        typical_id = list(remaining_ids)[0]
        mapping[typical_id] = "Typical Household"

        return dict(
            sorted(
                mapping.items(),
                key=lambda item: item[0]
            )
        )


if (
    "CLUSTER_NAMES" not in globals()
    or "CLUSTER_NAME_TO_ID" not in globals()
):
    CLUSTER_NAMES = infer_archetype_name_map(
        clean_fm,
        label_col="final_cluster"
    )

    CLUSTER_NAME_TO_ID = {
        cluster_name: cluster_id
        for cluster_id, cluster_name in CLUSTER_NAMES.items()
    }


if "Daytime-Active" not in CLUSTER_NAME_TO_ID:
    raise RuntimeError(
        "Daytime-Active could not be identified from the stored master clusters."
    )

daytime_cluster_id = CLUSTER_NAME_TO_ID["Daytime-Active"]


# ============================================================
# EXTRACT DAYTIME-ACTIVE HOUSEHOLDS
# ============================================================

da = (
    clean_fm.loc[
        clean_fm["final_cluster"] == daytime_cluster_id,
        CORE_FEATURES
    ]
    .copy()
)

print("=" * 70)
print("  DAYTIME-ACTIVE INTERNAL STRUCTURE ANALYSIS")
print(
    f"  Exploratory analysis within the "
    f"{len(da)}-household Daytime-Active archetype"
)
print("  The locked household-level K=4 solution is unchanged")
print("=" * 70)

print(
    f"\nDaytime-Active raw cluster ID: "
    f"{daytime_cluster_id}"
)

print(
    f"Daytime-Active population: "
    f"{len(da)} households"
)

if len(da) < 10:
    raise RuntimeError(
        "Too few Daytime-Active households for exploratory sub-clustering."
    )

if not np.isfinite(
    da.to_numpy(dtype=float)
).all():
    raise RuntimeError(
        "Daytime-Active feature data contain missing or infinite values."
    )


# ============================================================
# INTERNAL FEATURE VARIATION
# ============================================================

print(
    "\n=== FEATURE DISTRIBUTIONS WITHIN DAYTIME-ACTIVE ==="
)

display(
    da.describe().round(4)
)

print(
    "\n=== COEFFICIENT OF VARIATION PER FEATURE ==="
)

for column in CORE_FEATURES:
    mean_value = da[column].mean()

    if np.isclose(mean_value, 0.0):
        cv = np.nan
    else:
        cv = da[column].std() / mean_value

    print(
        f"  {column:<25}: "
        f"CV = {cv:.3f}"
    )


# ============================================================
# TEST K=2 AND K=3 INTERNAL SUB-CLUSTERS
# ============================================================

print("\n" + "=" * 70)
print("  TESTING K=2 AND K=3 INTERNAL SUB-CLUSTERS")
print("=" * 70)

daytime_scaler = StandardScaler()

X_da = daytime_scaler.fit_transform(
    da[CORE_FEATURES]
)

sub_labels_k2 = None
sub_silhouettes = {}
sub_cluster_sizes = {}

for k in [2, 3]:
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    sub_labels = model.fit_predict(
        X_da
    )

    silhouette = silhouette_score(
        X_da,
        sub_labels
    )

    sizes = sorted(
        pd.Series(sub_labels)
        .value_counts()
        .tolist(),
        reverse=True
    )

    sub_silhouettes[k] = float(silhouette)
    sub_cluster_sizes[k] = sizes

    print(
        f"\nK={k}: "
        f"Silhouette={silhouette:.3f}  "
        f"Sizes={sizes}"
    )

    profile_table = da.copy()
    profile_table["sub_cluster"] = sub_labels

    centroids = (
        profile_table
        .groupby("sub_cluster")[
            CORE_FEATURES
        ]
        .mean()
        .round(4)
    )

    print(
        centroids.to_string()
    )

    if k == 2:
        sub_labels_k2 = sub_labels.copy()


if sub_labels_k2 is None:
    raise RuntimeError(
        "K=2 sub-cluster labels were not created."
    )

print(f"""
Internal exploratory comparison:

  K=2 silhouette: {sub_silhouettes[2]:.3f}
  K=3 silhouette: {sub_silhouettes[3]:.3f}

K=3 achieved the slightly higher silhouette score. K=2 is retained only as
the simpler descriptive split for the supplementary administrative-data
comparison.

This exploratory analysis does not alter the locked K=4 household-level
clustering solution.
""")


# ============================================================
# LOAD SIX-HOUR ADMINISTRATIVE DATA
# ============================================================

print("\n" + "=" * 70)
print(
    "  ADMINISTRATIVE COMPARISON BY K=2 "
    "DAYTIME-ACTIVE SUB-CLUSTER"
)
print("=" * 70)

files = sorted(
    glob.glob(
        "smart_6hour_*.csv"
    )
)

if not files:
    print(
        "No smart_6hour_*.csv files were found. "
        "Administrative-data comparison was skipped."
    )

else:
    admin_frames = []

    for file_number, file_path in enumerate(files):
        frame = pd.read_csv(
            file_path,
            low_memory=False,
            usecols=[
                "household_ID",
                "SUB_TYPE",
                "DEPOT",
                "DATE",
                "TIME"
            ]
        )

        frame["_source_file"] = file_path
        frame["_file_order"] = file_number
        frame["_row_order"] = np.arange(len(frame))

        admin_frames.append(frame)

    raw_admin = pd.concat(
        admin_frames,
        ignore_index=True
    )


    # --------------------------------------------------------
    # CLEAN ADMINISTRATIVE FIELDS
    # --------------------------------------------------------

    raw_admin["household_ID"] = (
        raw_admin["household_ID"]
        .astype(str)
        .str.strip()
    )

    raw_admin["SUB_TYPE_clean"] = (
        raw_admin["SUB_TYPE"]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )

    raw_admin["DEPOT_clean"] = (
        raw_admin["DEPOT"]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )


    # --------------------------------------------------------
    # PARSE ADMINISTRATIVE RECORD DATETIME
    # --------------------------------------------------------

    datetime_text = (
        raw_admin["DATE"]
        .astype(str)
        .str.strip()
        + " "
        + raw_admin["TIME"]
        .astype(str)
        .str.strip()
    )

    raw_admin["record_datetime"] = pd.to_datetime(
        datetime_text,
        errors="coerce",
        dayfirst=True
    )

    parsed_timestamp_count = int(
        raw_admin["record_datetime"]
        .notna()
        .sum()
    )

    print("\nAdministrative timestamp parsing:")

    print(
        f"  Parsed timestamps: "
        f"{parsed_timestamp_count:,} / "
        f"{len(raw_admin):,}"
    )

    print(
        f"  Parsed date range: "
        f"{raw_admin['record_datetime'].min()} "
        f"to "
        f"{raw_admin['record_datetime'].max()}"
    )


    # --------------------------------------------------------
    # ADMINISTRATIVE METADATA CONSISTENCY AUDIT
    # --------------------------------------------------------

    metadata_consistency = (
        raw_admin
        .groupby("household_ID")
        .agg(
            unique_sub_types=(
                "SUB_TYPE_clean",
                lambda values: values.dropna().nunique()
            ),
            unique_depots=(
                "DEPOT_clean",
                lambda values: values.dropna().nunique()
            )
        )
    )

    sub_type_conflicts = int(
        (
            metadata_consistency[
                "unique_sub_types"
            ] > 1
        ).sum()
    )

    depot_conflicts = int(
        (
            metadata_consistency[
                "unique_depots"
            ] > 1
        ).sum()
    )

    print("\nAdministrative metadata consistency audit:")

    print(
        f"  Households with multiple SUB_TYPE values: "
        f"{sub_type_conflicts}"
    )

    print(
        f"  Households with multiple DEPOT values: "
        f"{depot_conflicts}"
    )


    # --------------------------------------------------------
    # DETERMINISTIC MODAL FALLBACK
    # --------------------------------------------------------

    def modal_non_null(series):
        available = (
            series
            .dropna()
            .astype(str)
            .str.strip()
        )

        available = available[
            available != ""
        ]

        if available.empty:
            return None

        counts = available.value_counts()

        tied_values = sorted(
            counts[
                counts == counts.max()
            ].index.tolist()
        )

        return tied_values[0]


    modal_subtype = (
        raw_admin
        .groupby("household_ID")[
            "SUB_TYPE_clean"
        ]
        .agg(modal_non_null)
        .rename("modal_subtype")
    )

    modal_depot = (
        raw_admin
        .groupby("household_ID")[
            "DEPOT_clean"
        ]
        .agg(modal_non_null)
        .rename("DEPOT")
    )


    # --------------------------------------------------------
    # TRUE CHRONOLOGICAL SUBTYPE
    # --------------------------------------------------------

    dated_subtype_rows = (
        raw_admin
        .dropna(
            subset=[
                "SUB_TYPE_clean",
                "record_datetime"
            ]
        )
        .sort_values(
            [
                "household_ID",
                "record_datetime",
                "_file_order",
                "_row_order"
            ]
        )
    )

    chronological_latest_subtype = (
        dated_subtype_rows
        .groupby("household_ID")
        .tail(1)
        .set_index("household_ID")[
            "SUB_TYPE_clean"
        ]
        .rename("chronological_latest_subtype")
    )


    # --------------------------------------------------------
    # LATEST SUBTYPE DURING THE GOLDEN WINDOW
    # --------------------------------------------------------

    golden_start = pd.Timestamp("2024-05-01")
    golden_end = pd.Timestamp("2024-12-01")

    golden_subtype_rows = dated_subtype_rows[
        (
            dated_subtype_rows[
                "record_datetime"
            ] >= golden_start
        )
        & (
            dated_subtype_rows[
                "record_datetime"
            ] < golden_end
        )
    ]

    golden_latest_subtype = (
        golden_subtype_rows
        .groupby("household_ID")
        .tail(1)
        .set_index("household_ID")[
            "SUB_TYPE_clean"
        ]
        .rename("golden_latest_subtype")
    )


    # --------------------------------------------------------
    # TEMPORALLY RELEVANT SUBTYPE
    #
    # Priority:
    # 1. Latest subtype during May-November 2024
    # 2. Latest chronologically parsed subtype
    # 3. Modal subtype only when no timestamped subtype exists
    # --------------------------------------------------------

    relevant_subtype = (
        golden_latest_subtype
        .combine_first(
            chronological_latest_subtype
        )
        .combine_first(
            modal_subtype
        )
        .rename("SUB_TYPE")
    )


    # --------------------------------------------------------
    # CREATE ONE ADMINISTRATIVE ROW PER HOUSEHOLD
    # --------------------------------------------------------

    admin = (
        pd.concat(
            [
                relevant_subtype,
                modal_depot
            ],
            axis=1
        )
        .reset_index()
    )

    assert (
        admin["household_ID"]
        .duplicated()
        .sum()
        == 0
    ), (
        "Administrative consolidation produced "
        "duplicate household IDs."
    )


    # --------------------------------------------------------
    # PREPARE DAYTIME-ACTIVE HOUSEHOLD IDS
    # --------------------------------------------------------

    da_reset = da.copy()
    da_reset["sub_cluster"] = sub_labels_k2
    da_reset = da_reset.reset_index()

    if "household_ID" not in da_reset.columns:
        first_column = da_reset.columns[0]

        da_reset = da_reset.rename(
            columns={
                first_column: "household_ID"
            }
        )

    if da_reset["household_ID"].duplicated().any():
        raise RuntimeError(
            "Duplicate Daytime-Active household IDs were found."
        )

    da_reset["household_ID"] = (
        da_reset["household_ID"]
        .astype(str)
        .str.strip()
    )

    admin["household_ID"] = (
        admin["household_ID"]
        .astype(str)
        .str.strip()
    )

    if admin["household_ID"].duplicated().any():
        raise RuntimeError(
            "Administrative metadata consolidation did not produce "
            "one row per household."
        )


    # --------------------------------------------------------
    # MERGE ADMINISTRATIVE INFORMATION
    # --------------------------------------------------------

    da_admin = pd.merge(
        da_reset[
            [
                "household_ID",
                "sub_cluster"
            ]
        ],
        admin,
        on="household_ID",
        how="left",
        validate="one_to_one"
    )

    if len(da_admin) != len(da_reset):
        raise RuntimeError(
            "Administrative merge changed the Daytime-Active population."
        )

    da_admin["sub_type_known"] = (
        da_admin["SUB_TYPE"].notna()
        & da_admin["SUB_TYPE"].astype(str).str.strip().ne("")
    )

    da_admin["three_phase"] = (
        da_admin["SUB_TYPE"]
        .astype("string")
        .str.contains(
            r"3\s*phase",
            case=False,
            regex=True,
            na=False
        )
    )


    # ========================================================
    # THREE-PHASE METER COMPARISON
    # ========================================================

    print(
        "\nThree-phase meter prevalence "
        "among households with known SUB_TYPE:"
    )

    three_phase_summary = []

    for sub_cluster in sorted(
        da_admin["sub_cluster"].unique()
    ):
        sub_data = da_admin[
            da_admin["sub_cluster"] == sub_cluster
        ]

        known_data = sub_data[
            sub_data["sub_type_known"]
        ]

        known_n = len(known_data)

        three_phase_n = int(
            known_data["three_phase"].sum()
        )

        rate = (
            three_phase_n / known_n * 100
            if known_n > 0
            else np.nan
        )

        three_phase_summary.append({
            "sub_cluster": int(sub_cluster),
            "total_households": len(sub_data),
            "known_sub_type": known_n,
            "three_phase_count": three_phase_n,
            "three_phase_percent": rate
        })

        if known_n > 0:
            print(
                f"  Sub-cluster {sub_cluster}: "
                f"{three_phase_n}/{known_n} known meters "
                f"({rate:.1f}% three-phase)"
            )
        else:
            print(
                f"  Sub-cluster {sub_cluster}: "
                "no known SUB_TYPE values"
            )

    three_phase_summary = (
        pd.DataFrame(
            three_phase_summary
        )
        .set_index("sub_cluster")
    )

    display(
        three_phase_summary.round(2)
    )


    # --------------------------------------------------------
    # FISHER'S EXACT TEST
    # --------------------------------------------------------

    known_meter_data = da_admin[
        da_admin["sub_type_known"]
    ].copy()

    meter_contingency = (
        pd.crosstab(
            known_meter_data["sub_cluster"],
            known_meter_data["three_phase"]
        )
        .reindex(
            columns=[
                False,
                True
            ],
            fill_value=0
        )
        .sort_index()
    )

    fisher_p = np.nan
    fisher_odds_ratio = np.nan

    if meter_contingency.shape == (2, 2):
        fisher_odds_ratio, fisher_p = fisher_exact(
            meter_contingency.to_numpy()
        )

        print(
            "\nExploratory Fisher's exact test for association "
            "between K=2 sub-cluster and three-phase-meter status:"
        )

        print(
            f"  Odds ratio: "
            f"{fisher_odds_ratio:.3f}"
        )

        print(
            f"  Unadjusted p-value: "
            f"{fisher_p:.6f}"
        )

    else:
        print(
            "\nFisher's exact test was not run because a complete "
            "2 x 2 table was not available."
        )


    # ========================================================
    # DEPOT DISTRIBUTION
    # ========================================================

    known_depot_data = da_admin[
        da_admin["DEPOT"].notna()
        & da_admin["DEPOT"].astype(str).str.strip().ne("")
    ].copy()

    print(
        "\nDEPOT distribution by sub-cluster "
        "(percentage within each sub-cluster):"
    )

    depot_percent = (
        pd.crosstab(
            known_depot_data["sub_cluster"],
            known_depot_data["DEPOT"],
            normalize="index"
        )
        * 100
    )

    if not depot_percent.empty:
        depot_percent = depot_percent.loc[
            :,
            (depot_percent > 5).any(axis=0)
        ]

        print(
            depot_percent
            .round(1)
            .to_string()
        )

    else:
        print(
            "No known DEPOT values were available."
        )

    print(
        "\nRaw DEPOT counts by sub-cluster:"
    )

    depot_counts = pd.crosstab(
        known_depot_data["sub_cluster"],
        known_depot_data["DEPOT"]
    )

    if depot_counts.empty:
        print(
            "No known DEPOT values were available."
        )
    else:
        print(
            depot_counts.to_string()
        )


    # ========================================================
    # FEATURE COMPARISON
    # ========================================================

    print("\n" + "=" * 70)
    print(
        "  FEATURE COMPARISON BETWEEN K=2 SUB-CLUSTERS"
    )
    print("=" * 70)

    daytime_profiles = da.copy()
    daytime_profiles["sub_cluster"] = sub_labels_k2

    summary = (
        daytime_profiles
        .groupby("sub_cluster")[
            CORE_FEATURES
        ]
        .agg([
            "mean",
            "median"
        ])
        .round(3)
    )

    print(
        summary.to_string()
    )


    # ========================================================
    # CONSERVATIVE INTERPRETATION
    # ========================================================

    print("\n" + "=" * 70)
    print("  CONSERVATIVE INTERPRETATION")
    print("=" * 70)

    valid_rates = (
        three_phase_summary[
            "three_phase_percent"
        ]
        .dropna()
    )

    if len(valid_rates) == 2:
        higher_three_phase_cluster = valid_rates.idxmax()
        lower_three_phase_cluster = valid_rates.idxmin()

        higher_rate = valid_rates.loc[
            higher_three_phase_cluster
        ]

        lower_rate = valid_rates.loc[
            lower_three_phase_cluster
        ]

        rate_text = f"""
The two exploratory K=2 sub-clusters differed in observed three-phase-meter
prevalence:

  Sub-cluster {higher_three_phase_cluster}: {higher_rate:.1f}%
  Sub-cluster {lower_three_phase_cluster}: {lower_rate:.1f}%
"""
    else:
        rate_text = """
A complete comparison of three-phase-meter prevalence could not be produced
because sufficient known SUB_TYPE information was unavailable.
"""

    significance_text = ""

    if np.isfinite(fisher_p):
        if fisher_p < 0.05:
            significance_text = f"""
Using the temporally aligned meter subtype, Fisher's exact test produced an
unadjusted p-value of {fisher_p:.4f}. This borderline result suggests a
possible association between exploratory sub-cluster membership and
three-phase-meter status.

However, the result is sensitive to the treatment of one household with a
documented subtype change. It is therefore interpreted as exploratory rather
than confirmatory evidence. It does not establish causation, premise type or
occupant characteristics.
"""
        else:
            significance_text = f"""
The exploratory Fisher's exact test produced an unadjusted p-value of
{fisher_p:.4f}. This did not provide evidence of an association between
sub-cluster membership and three-phase-meter status at the conventional
5% level.
"""

    print(
        rate_text
        + significance_text
        + """
The sub-clusters also differ in their electricity-use profiles and
administrative-location distributions. These findings support descriptive
internal heterogeneity within the Daytime-Active archetype.

However, meter subtype and depot do not establish whether a premise is
commercial or residential, and they do not identify the occupation,
employment status, age or caregiving role of household occupants. The
available data therefore do not support explanations involving commercial
premises, work-from-home professionals, retirees or caregivers.

The locked household-level K=4 clustering solution remains unchanged.
"""
    )


In [ ]:
# ============================================================
# EXACT SUB_TYPE RESOLUTION AUDIT
# Compares:
# 1. Old row-order last value
# 2. Modal value
# 3. True chronological latest value
# 4. Latest value during the May-November 2024 golden window
# ============================================================

import pandas as pd
import numpy as np
import glob

from scipy.stats import fisher_exact


# ------------------------------------------------------------
# READ RAW ADMINISTRATIVE RECORDS WITH DATE AND TIME
# ------------------------------------------------------------

audit_files = sorted(
    glob.glob("smart_6hour_*.csv")
)

if not audit_files:
    raise FileNotFoundError(
        "No smart_6hour_*.csv files were found."
    )

audit_parts = []

for file_number, file_path in enumerate(audit_files):

    temp = pd.read_csv(
        file_path,
        usecols=[
            "household_ID",
            "SUB_TYPE",
            "DEPOT",
            "DATE",
            "TIME"
        ],
        low_memory=False
    )

    temp["_source_file"] = file_path
    temp["_file_order"] = file_number
    temp["_row_order"] = np.arange(len(temp))

    audit_parts.append(temp)

raw_subtype_audit = pd.concat(
    audit_parts,
    ignore_index=True
)

raw_subtype_audit["household_ID"] = (
    raw_subtype_audit["household_ID"]
    .astype(str)
    .str.strip()
)

raw_subtype_audit["SUB_TYPE_clean"] = (
    raw_subtype_audit["SUB_TYPE"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
)

raw_subtype_audit["DEPOT_clean"] = (
    raw_subtype_audit["DEPOT"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
)


# ------------------------------------------------------------
# PARSE TRUE RECORD DATETIME
# ------------------------------------------------------------

datetime_text = (
    raw_subtype_audit["DATE"]
    .astype(str)
    .str.strip()
    + " "
    + raw_subtype_audit["TIME"]
    .astype(str)
    .str.strip()
)

raw_subtype_audit["record_datetime"] = pd.to_datetime(
    datetime_text,
    errors="coerce",
    dayfirst=True
)

print(
    "Parsed administrative timestamps:",
    raw_subtype_audit["record_datetime"]
    .notna()
    .sum(),
    "/",
    len(raw_subtype_audit)
)

print(
    "Administrative date range:",
    raw_subtype_audit["record_datetime"].min(),
    "to",
    raw_subtype_audit["record_datetime"].max()
)


# ------------------------------------------------------------
# CONSOLIDATION FUNCTIONS
# ------------------------------------------------------------

def row_order_last(series):

    available = series.dropna()

    if available.empty:
        return None

    return available.iloc[-1]


def deterministic_mode(series):

    available = series.dropna()

    if available.empty:
        return None

    counts = available.value_counts()

    tied = sorted(
        counts[
            counts == counts.max()
        ].index.tolist()
    )

    return tied[0]


# Old rule: last value in concatenated file/row order
row_last_subtype = (
    raw_subtype_audit
    .groupby("household_ID")[
        "SUB_TYPE_clean"
    ]
    .agg(row_order_last)
    .rename("row_order_last")
)


# Modal rule
modal_subtype = (
    raw_subtype_audit
    .groupby("household_ID")[
        "SUB_TYPE_clean"
    ]
    .agg(deterministic_mode)
    .rename("modal_value")
)


# True latest chronological value
chronological_rows = (
    raw_subtype_audit
    .dropna(
        subset=[
            "SUB_TYPE_clean",
            "record_datetime"
        ]
    )
    .sort_values(
        [
            "household_ID",
            "record_datetime",
            "_file_order",
            "_row_order"
        ]
    )
)

chronological_latest = (
    chronological_rows
    .groupby("household_ID")
    .tail(1)
    .set_index("household_ID")[
        "SUB_TYPE_clean"
    ]
    .rename("chronological_latest")
)


# Latest value inside the clustering golden window
golden_start = pd.Timestamp("2024-05-01")
golden_end = pd.Timestamp("2024-12-01")

golden_rows = chronological_rows[
    (
        chronological_rows["record_datetime"]
        >= golden_start
    )
    & (
        chronological_rows["record_datetime"]
        < golden_end
    )
]

golden_latest = (
    golden_rows
    .groupby("household_ID")
    .tail(1)
    .set_index("household_ID")[
        "SUB_TYPE_clean"
    ]
    .rename("golden_window_latest")
)


# Use chronological latest as fallback where no
# golden-window record is available.
relevant_latest = (
    golden_latest
    .combine_first(
        chronological_latest
    )
    .rename("relevant_latest")
)


# ------------------------------------------------------------
# BUILD HOUSEHOLD-LEVEL COMPARISON TABLE
# ------------------------------------------------------------

subtype_comparison = pd.concat(
    [
        row_last_subtype,
        modal_subtype,
        chronological_latest,
        golden_latest,
        relevant_latest
    ],
    axis=1
)

subtype_comparison["unique_subtype_count"] = (
    raw_subtype_audit
    .groupby("household_ID")[
        "SUB_TYPE_clean"
    ]
    .nunique()
)

subtype_comparison["has_conflict"] = (
    subtype_comparison[
        "unique_subtype_count"
    ] > 1
)


# ------------------------------------------------------------
# RESTRICT TO DAYTIME-ACTIVE K=2 SUBCLUSTERS
# ------------------------------------------------------------

daytime_membership = (
    da_reset[
        [
            "household_ID",
            "sub_cluster"
        ]
    ]
    .copy()
)

daytime_membership["household_ID"] = (
    daytime_membership["household_ID"]
    .astype(str)
    .str.strip()
)

daytime_subtype_comparison = (
    daytime_membership
    .merge(
        subtype_comparison.reset_index(),
        on="household_ID",
        how="left",
        validate="one_to_one"
    )
)


# ------------------------------------------------------------
# IDENTIFY EXACT HOUSEHOLDS WHOSE RESULT CHANGES
# ------------------------------------------------------------

method_columns = [
    "row_order_last",
    "modal_value",
    "chronological_latest",
    "golden_window_latest",
    "relevant_latest"
]

daytime_subtype_comparison[
    "methods_disagree"
] = (
    daytime_subtype_comparison[
        method_columns
    ]
    .astype("string")
    .nunique(
        axis=1,
        dropna=True
    )
    > 1
)

changed_daytime_households = (
    daytime_subtype_comparison[
        daytime_subtype_comparison[
            "methods_disagree"
        ]
    ]
    .sort_values(
        [
            "sub_cluster",
            "household_ID"
        ]
    )
)

print(
    "\nDaytime-Active households for which "
    "SUB_TYPE resolution methods disagree:",
    len(changed_daytime_households)
)

display(
    changed_daytime_households[
        [
            "household_ID",
            "sub_cluster",
            "unique_subtype_count",
            "row_order_last",
            "modal_value",
            "chronological_latest",
            "golden_window_latest",
            "relevant_latest"
        ]
    ]
)


# ------------------------------------------------------------
# SHOW RAW CHRONOLOGICAL RECORDS FOR THOSE HOUSEHOLDS
# ------------------------------------------------------------

changed_ids = set(
    changed_daytime_households[
        "household_ID"
    ]
)

changed_raw_records = (
    raw_subtype_audit[
        raw_subtype_audit[
            "household_ID"
        ].isin(changed_ids)
    ]
    [
        [
            "household_ID",
            "record_datetime",
            "SUB_TYPE_clean",
            "DEPOT_clean",
            "_source_file",
            "_row_order"
        ]
    ]
    .sort_values(
        [
            "household_ID",
            "record_datetime",
            "_source_file",
            "_row_order"
        ]
    )
)

print(
    "\nRaw chronological records for "
    "changed Daytime-Active households:"
)

display(changed_raw_records)


# ------------------------------------------------------------
# DISPLAY VALUE COUNTS FOR EACH CHANGED HOUSEHOLD
# ------------------------------------------------------------

value_count_rows = []

for household_id in sorted(changed_ids):

    household_values = (
        raw_subtype_audit[
            raw_subtype_audit[
                "household_ID"
            ] == household_id
        ][
            "SUB_TYPE_clean"
        ]
        .dropna()
        .value_counts()
    )

    for subtype, count in household_values.items():

        value_count_rows.append({
            "household_ID":
                household_id,
            "SUB_TYPE":
                subtype,
            "record_count":
                int(count)
        })

subtype_value_counts = pd.DataFrame(
    value_count_rows
)

print(
    "\nSUB_TYPE frequency counts for "
    "changed households:"
)

display(subtype_value_counts)


# ------------------------------------------------------------
# RECALCULATE THREE-PHASE TABLE AND FISHER TEST
# UNDER EVERY RESOLUTION RULE
# ------------------------------------------------------------

def is_three_phase(series):

    return (
        series
        .astype("string")
        .str.contains(
            r"3\s*phase",
            case=False,
            regex=True,
            na=False
        )
    )


def evaluate_resolution_method(
    comparison_table,
    method_column
):

    evaluation = comparison_table[
        [
            "household_ID",
            "sub_cluster",
            method_column
        ]
    ].copy()

    evaluation = evaluation[
        evaluation[method_column]
        .notna()
    ]

    evaluation["three_phase"] = (
        is_three_phase(
            evaluation[method_column]
        )
    )

    contingency = (
        pd.crosstab(
            evaluation["sub_cluster"],
            evaluation["three_phase"]
        )
        .reindex(
            index=[0, 1],
            columns=[False, True],
            fill_value=0
        )
    )

    odds_ratio, p_value = fisher_exact(
        contingency.to_numpy()
    )

    return {
        "method":
            method_column,
        "subcluster_0_three_phase":
            int(contingency.loc[0, True]),
        "subcluster_0_total":
            int(contingency.loc[0].sum()),
        "subcluster_1_three_phase":
            int(contingency.loc[1, True]),
        "subcluster_1_total":
            int(contingency.loc[1].sum()),
        "odds_ratio":
            float(odds_ratio),
        "fisher_p":
            float(p_value)
    }


method_results = pd.DataFrame(
    [
        evaluate_resolution_method(
            daytime_subtype_comparison,
            method
        )
        for method in method_columns
    ]
)

print(
    "\nThree-phase and Fisher results "
    "under each SUB_TYPE resolution method:"
)

display(
    method_results.round(6)
)


# ------------------------------------------------------------
# CONFLICT-EXCLUDED SENSITIVITY ANALYSIS
# ------------------------------------------------------------

conflict_free_daytime = (
    daytime_subtype_comparison[
        ~daytime_subtype_comparison[
            "has_conflict"
        ].fillna(False)
    ]
    .copy()
)

conflict_free_result = evaluate_resolution_method(
    conflict_free_daytime,
    "relevant_latest"
)

print(
    "\nConflict-excluded sensitivity result:"
)

display(
    pd.DataFrame(
        [conflict_free_result]
    ).round(6)
)